# Exploratory Data Analysis: Facebook Political Ads TEXT Dataset

This notebook provides a comprehensive text analysis of the Facebook political ads text dataset, focusing on:
- Text content analysis
- NLP features extraction
- Manipulation indicators in text
- Sentiment and emotional analysis
- Topic modeling and keyword extraction

## Dataset Overview
- **Source**: Facebook Ad Library 2022 - Text Content
- **Purpose**: Analyze ad text for manipulation detection
- **Connection**: Links to metadata dataset via `ad_id`


In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
import warnings
import re
from collections import Counter
from wordcloud import WordCloud
import plotly.express as px
import plotly.graph_objects as go

# NLP libraries
try:
    from textblob import TextBlob
    TEXTBLOB_AVAILABLE = True
except ImportError:
    TEXTBLOB_AVAILABLE = False
    print("TextBlob not available. Install with: pip install textblob")

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_colwidth', 200)

# Set plotting style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (14, 8)
warnings.filterwarnings('ignore')

print("Libraries imported successfully!")


In [ ]:
# Load the text dataset
data_path = '../data/fb_2022_adid_text_102424.csv'
df_text = pd.read_csv(data_path, low_memory=False)

print(f"Dataset shape: {df_text.shape}")
print(f"Memory usage: {df_text.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
print(f"\nColumns: {len(df_text.columns)}")
print(f"Rows: {len(df_text):,}")


In [ ]:
# Display basic information
print("=" * 80)
print("DATASET OVERVIEW")
print("=" * 80)
print(f"\nColumn names:")
for i, col in enumerate(df_text.columns, 1):
    print(f"{i:2d}. {col}")

print(f"\n\nFirst few rows:")
df_text.head()


In [ ]:
# Missing values analysis for text columns
print("=" * 80)
print("MISSING VALUES ANALYSIS - TEXT COLUMNS")
print("=" * 80)

text_columns = [
    'ad_creative_body',
    'ad_creative_link_title',
    'ad_creative_link_description',
    'ad_creative_link_caption',
    'disclaimer',
    'google_asr_text',
    'aws_ocr_text_img',
    'aws_ocr_text_vid'
]

missing_data = []
for col in text_columns:
    if col in df_text.columns:
        missing = df_text[col].isna().sum()
        non_empty = df_text[col].notna() & (df_text[col].astype(str).str.strip() != '')
        non_empty_count = non_empty.sum()
        missing_data.append({
            'Column': col,
            'Missing': missing,
            'Missing_Pct': (missing / len(df_text)) * 100,
            'Non_Empty': non_empty_count,
            'Non_Empty_Pct': (non_empty_count / len(df_text)) * 100
        })

missing_df = pd.DataFrame(missing_data)
print(missing_df.to_string(index=False))

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

axes[0].barh(missing_df['Column'], missing_df['Missing_Pct'], color='coral', alpha=0.7)
axes[0].set_xlabel('Missing Percentage (%)', fontsize=11)
axes[0].set_title('Missing Values by Column', fontsize=12, fontweight='bold')
axes[0].grid(True, alpha=0.3, axis='x')

axes[1].barh(missing_df['Column'], missing_df['Non_Empty_Pct'], color='steelblue', alpha=0.7)
axes[1].set_xlabel('Non-Empty Percentage (%)', fontsize=11)
axes[1].set_title('Non-Empty Values by Column', fontsize=12, fontweight='bold')
axes[1].grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.show()


In [ ]:
# Create combined text field for analysis
def combine_text(row):
    """Combine all text sources into one field"""
    texts = []
    for col in text_columns:
        if col in df_text.columns:
            val = row.get(col)
            if pd.notna(val) and str(val).strip() != '':
                texts.append(str(val).strip())
    return ' '.join(texts) if texts else ''

df_text['combined_text'] = df_text.apply(combine_text, axis=1)
df_text['combined_text_length'] = df_text['combined_text'].str.len()

print("=" * 80)
print("COMBINED TEXT STATISTICS")
print("=" * 80)
print(f"\nRows with any text: {df_text['combined_text_length'].gt(0).sum():,}")
print(f"Rows with text > 50 chars: {(df_text['combined_text_length'] > 50).sum():,}")
print(f"Rows with text > 100 chars: {(df_text['combined_text_length'] > 100).sum():,}")
print(f"Rows with text > 500 chars: {(df_text['combined_text_length'] > 500).sum():,}")
print(f"\nText length statistics:")
print(df_text['combined_text_length'].describe())

# Visualize text length distribution
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

non_zero = df_text[df_text['combined_text_length'] > 0]['combined_text_length']
axes[0].hist(non_zero, bins=50, edgecolor='black', alpha=0.7, color='steelblue')
axes[0].set_xlabel('Text Length (characters)', fontsize=11)
axes[0].set_ylabel('Frequency', fontsize=11)
axes[0].set_title('Text Length Distribution (Non-Zero)', fontsize=12, fontweight='bold')
axes[0].set_yscale('log')
axes[0].grid(True, alpha=0.3)

axes[1].hist(non_zero[non_zero < 2000], bins=50, edgecolor='black', alpha=0.7, color='coral')
axes[1].set_xlabel('Text Length (characters)', fontsize=11)
axes[1].set_ylabel('Frequency', fontsize=11)
axes[1].set_title('Text Length Distribution (< 2000 chars)', fontsize=12, fontweight='bold')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


In [ ]:
# Sample some actual ad text content
print("=" * 80)
print("SAMPLE AD TEXT CONTENT")
print("=" * 80)

# Get rows with substantial text
substantial_text = df_text[df_text['combined_text_length'] > 100].head(10)

for idx, (row_idx, row) in enumerate(substantial_text.iterrows(), 1):
    print(f"\n{'='*80}")
    print(f"Example {idx} (Row {row_idx}, Length: {row['combined_text_length']} chars)")
    print(f"{'='*80}")
    print(f"\nAd ID: {row.get('ad_id', 'N/A')}")
    print(f"Page Name: {row.get('page_name', 'N/A')}")
    
    if pd.notna(row.get('ad_creative_body')):
        print(f"\nAd Creative Body:")
        print(f"{str(row['ad_creative_body'])[:500]}")
    
    if pd.notna(row.get('ad_creative_link_title')):
        print(f"\nLink Title: {row['ad_creative_link_title']}")
    
    if pd.notna(row.get('disclaimer')):
        print(f"\nDisclaimer: {str(row['disclaimer'])[:200]}")
    
    if pd.notna(row.get('aws_ocr_text_img')):
        print(f"\nOCR Text (Image): {str(row['aws_ocr_text_img'])[:200]}")
    
    if pd.notna(row.get('google_asr_text')):
        print(f"\nASR Text (Video): {str(row['google_asr_text'])[:200]}")


In [ ]:
# Text statistics by column
print("=" * 80)
print("TEXT STATISTICS BY COLUMN")
print("=" * 80)

text_stats = []
for col in text_columns:
    if col in df_text.columns:
        non_empty = df_text[col].dropna()
        non_empty = non_empty[non_empty.astype(str).str.strip() != '']
        
        if len(non_empty) > 0:
            lengths = non_empty.astype(str).str.len()
            text_stats.append({
                'Column': col,
                'Count': len(non_empty),
                'Mean_Length': lengths.mean(),
                'Median_Length': lengths.median(),
                'Min_Length': lengths.min(),
                'Max_Length': lengths.max(),
                'Std_Length': lengths.std()
            })

stats_df = pd.DataFrame(text_stats)
print(stats_df.to_string(index=False))

# Visualize
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

axes[0, 0].barh(stats_df['Column'], stats_df['Count'], color='steelblue', alpha=0.7)
axes[0, 0].set_xlabel('Count', fontsize=11)
axes[0, 0].set_title('Non-Empty Text Count by Column', fontsize=12, fontweight='bold')
axes[0, 0].grid(True, alpha=0.3, axis='x')

axes[0, 1].barh(stats_df['Column'], stats_df['Mean_Length'], color='coral', alpha=0.7)
axes[0, 1].set_xlabel('Mean Length (chars)', fontsize=11)
axes[0, 1].set_title('Mean Text Length by Column', fontsize=12, fontweight='bold')
axes[0, 1].grid(True, alpha=0.3, axis='x')

axes[1, 0].barh(stats_df['Column'], stats_df['Median_Length'], color='green', alpha=0.7)
axes[1, 0].set_xlabel('Median Length (chars)', fontsize=11)
axes[1, 0].set_title('Median Text Length by Column', fontsize=12, fontweight='bold')
axes[1, 0].grid(True, alpha=0.3, axis='x')

axes[1, 1].barh(stats_df['Column'], stats_df['Max_Length'], color='purple', alpha=0.7)
axes[1, 1].set_xlabel('Max Length (chars)', fontsize=11)
axes[1, 1].set_title('Max Text Length by Column', fontsize=12, fontweight='bold')
axes[1, 1].grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.show()


In [ ]:
# Basic NLP features extraction
print("=" * 80)
print("BASIC NLP FEATURES")
print("=" * 80)

def extract_nlp_features(text):
    """Extract basic NLP features from text"""
    if pd.isna(text) or str(text).strip() == '':
        return {
            'word_count': 0,
            'char_count': 0,
            'sentence_count': 0,
            'avg_word_length': 0,
            'uppercase_ratio': 0,
            'exclamation_count': 0,
            'question_count': 0,
            'has_url': 0,
            'has_emoji': 0
        }
    
    text_str = str(text)
    words = text_str.split()
    sentences = re.split(r'[.!?]+', text_str)
    
    return {
        'word_count': len(words),
        'char_count': len(text_str),
        'sentence_count': len([s for s in sentences if s.strip()]),
        'avg_word_length': np.mean([len(w) for w in words]) if words else 0,
        'uppercase_ratio': sum(1 for c in text_str if c.isupper()) / len(text_str) if text_str else 0,
        'exclamation_count': text_str.count('!'),
        'question_count': text_str.count('?'),
        'has_url': 1 if re.search(r'http[s]?://(?:[a-zA-Z]|[0-9]|[$-_@.&+]|[!*\\\\(\\\\),]|(?:%[0-9a-fA-F][0-9a-fA-F]))+', text_str) else 0,
        'has_emoji': 1 if re.search(r'[\\U0001F600-\\U0001F64F\\U0001F300-\\U0001F5FF\\U0001F680-\\U0001F6FF\\U0001F1E0-\\U0001F1FF\\U00002702-\\U000027B0\\U000024C2-\\U0001F251]+', text_str) else 0
    }

# Extract features for combined text (sample for performance)
sample_size = min(50000, len(df_text))
sample_df = df_text[df_text['combined_text_length'] > 0].head(sample_size).copy()

print(f"Extracting NLP features for {len(sample_df):,} rows with text...")
nlp_features = sample_df['combined_text'].apply(extract_nlp_features)
nlp_df = pd.DataFrame(list(nlp_features))

# Add to sample dataframe
for col in nlp_df.columns:
    sample_df[f'nlp_{col}'] = nlp_df[col]

print("\nNLP Features Statistics:")
print(nlp_df.describe())

# Visualize NLP features
fig, axes = plt.subplots(3, 3, figsize=(18, 12))

features_to_plot = ['word_count', 'char_count', 'sentence_count', 'avg_word_length', 
                    'uppercase_ratio', 'exclamation_count', 'question_count', 'has_url', 'has_emoji']

for idx, feat in enumerate(features_to_plot):
    ax = axes[idx // 3, idx % 3]
    if feat in nlp_df.columns:
        data = nlp_df[feat][nlp_df[feat] > 0] if feat not in ['has_url', 'has_emoji'] else nlp_df[feat]
        ax.hist(data, bins=30, edgecolor='black', alpha=0.7)
        ax.set_xlabel(feat.replace('_', ' ').title(), fontsize=10)
        ax.set_ylabel('Frequency', fontsize=10)
        ax.set_title(f'{feat.replace("_", " ").title()} Distribution', fontsize=11, fontweight='bold')
        if feat not in ['has_url', 'has_emoji', 'uppercase_ratio']:
            ax.set_yscale('log')
        ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


In [ ]:
# Sentiment analysis (if TextBlob is available)
print("=" * 80)
print("SENTIMENT ANALYSIS")
print("=" * 80)

if TEXTBLOB_AVAILABLE:
    def get_sentiment(text):
        """Get sentiment polarity and subjectivity"""
        if pd.isna(text) or str(text).strip() == '':
            return {'polarity': 0.0, 'subjectivity': 0.0}
        try:
            blob = TextBlob(str(text))
            return {'polarity': blob.sentiment.polarity, 'subjectivity': blob.sentiment.subjectivity}
        except:
            return {'polarity': 0.0, 'subjectivity': 0.0}
    
    # Analyze sentiment for sample
    print(f"Analyzing sentiment for {len(sample_df):,} rows...")
    sentiment_features = sample_df['combined_text'].apply(get_sentiment)
    sentiment_df = pd.DataFrame(list(sentiment_features))
    
    for col in sentiment_df.columns:
        sample_df[f'sentiment_{col}'] = sentiment_df[col]
    
    print("\nSentiment Statistics:")
    print(sentiment_df.describe())
    
    # Visualize sentiment
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    
    axes[0].hist(sentiment_df['polarity'], bins=50, edgecolor='black', alpha=0.7, color='steelblue')
    axes[0].axvline(x=0, color='red', linestyle='--', linewidth=2, label='Neutral')
    axes[0].set_xlabel('Polarity (-1 = Negative, +1 = Positive)', fontsize=11)
    axes[0].set_ylabel('Frequency', fontsize=11)
    axes[0].set_title('Sentiment Polarity Distribution', fontsize=12, fontweight='bold')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    axes[1].hist(sentiment_df['subjectivity'], bins=50, edgecolor='black', alpha=0.7, color='coral')
    axes[1].set_xlabel('Subjectivity (0 = Objective, 1 = Subjective)', fontsize=11)
    axes[1].set_ylabel('Frequency', fontsize=11)
    axes[1].set_title('Sentiment Subjectivity Distribution', fontsize=12, fontweight='bold')
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Sentiment categories
    sample_df['sentiment_category'] = pd.cut(
        sample_df['sentiment_polarity'],
        bins=[-1, -0.1, 0.1, 1],
        labels=['Negative', 'Neutral', 'Positive']
    )
    
    print("\nSentiment Category Distribution:")
    print(sample_df['sentiment_category'].value_counts())
else:
    print("TextBlob not available. Install with: pip install textblob")
    print("Skipping sentiment analysis.")


In [ ]:
# Most common words analysis
print("=" * 80)
print("MOST COMMON WORDS ANALYSIS")
print("=" * 80)

def clean_text_for_words(text):
    """Clean text for word frequency analysis"""
    if pd.isna(text) or str(text).strip() == '':
        return []
    
    # Convert to lowercase and remove special characters
    text = str(text).lower()
    text = re.sub(r'[^a-z\\s]', ' ', text)
    
    # Common stopwords (basic list)
    stopwords = {'the', 'a', 'an', 'and', 'or', 'but', 'in', 'on', 'at', 'to', 'for', 
                 'of', 'with', 'by', 'is', 'are', 'was', 'were', 'be', 'been', 'being',
                 'have', 'has', 'had', 'do', 'does', 'did', 'will', 'would', 'should',
                 'could', 'may', 'might', 'must', 'can', 'this', 'that', 'these', 'those',
                 'i', 'you', 'he', 'she', 'it', 'we', 'they', 'me', 'him', 'her', 'us', 'them'}
    
    words = [w for w in text.split() if len(w) > 2 and w not in stopwords]
    return words

# Get words from sample
print(f"Extracting words from {len(sample_df):,} rows...")
all_words = []
for text in sample_df['combined_text']:
    words = clean_text_for_words(text)
    all_words.extend(words)

word_counts = Counter(all_words)

print(f"\nTotal unique words: {len(word_counts):,}")
print(f"Total word occurrences: {len(all_words):,}")
print(f"\nTop 50 most common words:")
for word, count in word_counts.most_common(50):
    print(f"  {word:20s}: {count:6,}")

# Visualize top words
top_words = dict(word_counts.most_common(30))

fig, ax = plt.subplots(figsize=(12, 8))
words_list = list(top_words.keys())
counts_list = list(top_words.values())
ax.barh(words_list, counts_list, color='steelblue', alpha=0.7)
ax.set_xlabel('Frequency', fontsize=11)
ax.set_title('Top 30 Most Common Words', fontsize=12, fontweight='bold')
ax.invert_yaxis()
ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

# Word cloud (if enough words)
if len(word_counts) > 0:
    try:
        wordcloud = WordCloud(width=800, height=400, background_color='white').generate_from_frequencies(word_counts)
        plt.figure(figsize=(16, 8))
        plt.imshow(wordcloud, interpolation='bilinear')
        plt.axis('off')
        plt.title('Word Cloud of Ad Text', fontsize=16, fontweight='bold', pad=20)
        plt.tight_layout()
        plt.show()
    except Exception as e:
        print(f"Could not generate word cloud: {e}")


In [ ]:
# Manipulation indicators in text
print("=" * 80)
print("MANIPULATION INDICATORS IN TEXT")
print("=" * 80)

# Define manipulation patterns
manipulation_patterns = {
    'urgency': [r'\\b(urgent|immediately|now|hurry|limited time|act now|don\\'t wait)\\b',
                r'\\b(deadline|expires|ending soon|last chance)\\b'],
    'fear_appeal': [r'\\b(danger|threat|warning|alert|dangerous|fear|scary)\\b',
                    r'\\b(attack|destroy|ruin|disaster|catastrophe)\\b'],
    'exaggeration': [r'\\b(guaranteed|100%|always|never|best|worst|amazing|incredible)\\b',
                     r'\\b(revolutionary|breakthrough|miracle|unbelievable)\\b'],
    'emotional_manipulation': [r'\\b(heartbreaking|devastating|shocking|outrageous)\\b',
                               r'\\b(you must|you need to|you have to|you should)\\b'],
    'false_authority': [r'\\b(experts say|studies show|research proves|scientists agree)\\b',
                        r'\\b(doctors recommend|authorities warn)\\b'],
    'scarcity': [r'\\b(limited|exclusive|rare|only|few left|running out)\\b',
                 r'\\b(while supplies last|one time offer)\\b'],
    'social_proof': [r'\\b(everyone|millions|thousands|join|don\\'t be left out)\\b',
                    r'\\b(most popular|trending|viral)\\b'],
    'call_to_action_aggressive': [r'\\b(click now|sign up now|donate now|vote now)\\b',
                                   r'\\b(join today|act today|do it now)\\b']
}

def count_manipulation_patterns(text):
    """Count manipulation patterns in text"""
    if pd.isna(text) or str(text).strip() == '':
        return {key: 0 for key in manipulation_patterns.keys()}
    
    text_lower = str(text).lower()
    counts = {}
    
    for pattern_type, patterns in manipulation_patterns.items():
        count = 0
        for pattern in patterns:
            matches = re.findall(pattern, text_lower, re.IGNORECASE)
            count += len(matches)
        counts[pattern_type] = count
    
    return counts

# Analyze manipulation patterns
print(f"Analyzing manipulation patterns in {len(sample_df):,} rows...")
manipulation_features = sample_df['combined_text'].apply(count_manipulation_patterns)
manipulation_df = pd.DataFrame(list(manipulation_features))

# Add to sample dataframe
for col in manipulation_df.columns:
    sample_df[f'manip_{col}'] = manipulation_df[col]

print("\nManipulation Pattern Statistics:")
print(manipulation_df.describe())

# Count ads with manipulation patterns
print("\nAds with manipulation patterns (>0 occurrences):")
for col in manipulation_df.columns:
    count = (manipulation_df[col] > 0).sum()
    pct = (count / len(manipulation_df)) * 100
    print(f"  {col:30s}: {count:6,} ({pct:5.2f}%)")

# Visualize
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
axes = axes.flatten()

for idx, col in enumerate(manipulation_df.columns):
    data = manipulation_df[col][manipulation_df[col] > 0]
    if len(data) > 0:
        axes[idx].hist(data, bins=20, edgecolor='black', alpha=0.7, color='coral')
        axes[idx].set_xlabel('Count', fontsize=9)
        axes[idx].set_ylabel('Frequency', fontsize=9)
        axes[idx].set_title(col.replace('_', ' ').title(), fontsize=10, fontweight='bold')
        axes[idx].set_yscale('log')
        axes[idx].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


In [ ]:
# Political keywords analysis
print("=" * 80)
print("POLITICAL KEYWORDS ANALYSIS")
print("=" * 80)

political_keywords = {
    'election': ['vote', 'voting', 'election', 'ballot', 'poll', 'candidate', 'campaign'],
    'party': ['democrat', 'republican', 'liberal', 'conservative', 'party', 'gop', 'dems'],
    'policy': ['policy', 'legislation', 'bill', 'law', 'congress', 'senate', 'house'],
    'issues': ['healthcare', 'immigration', 'economy', 'tax', 'education', 'climate', 'abortion'],
    'action': ['donate', 'contribute', 'support', 'endorse', 'oppose', 'protest', 'rally'],
    'emotion': ['fight', 'battle', 'war', 'defend', 'protect', 'save', 'stop', 'defeat']
}

def count_political_keywords(text):
    """Count political keywords in text"""
    if pd.isna(text) or str(text).strip() == '':
        return {key: 0 for key in political_keywords.keys()}
    
    text_lower = str(text).lower()
    counts = {}
    
    for category, keywords in political_keywords.items():
        count = sum(1 for keyword in keywords if keyword in text_lower)
        counts[category] = count
    
    return counts

# Analyze political keywords
print(f"Analyzing political keywords in {len(sample_df):,} rows...")
political_features = sample_df['combined_text'].apply(count_political_keywords)
political_df = pd.DataFrame(list(political_features))

# Add to sample dataframe
for col in political_df.columns:
    sample_df[f'political_{col}'] = political_df[col]

print("\nPolitical Keyword Statistics:")
print(political_df.describe())

# Count ads with political keywords
print("\nAds with political keywords (>0 occurrences):")
for col in political_df.columns:
    count = (political_df[col] > 0).sum()
    pct = (count / len(political_df)) * 100
    print(f"  {col:30s}: {count:6,} ({pct:5.2f}%)")

# Visualize
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for idx, col in enumerate(political_df.columns):
    data = political_df[col][political_df[col] > 0]
    if len(data) > 0:
        axes[idx].hist(data, bins=20, edgecolor='black', alpha=0.7, color='steelblue')
        axes[idx].set_xlabel('Count', fontsize=10)
        axes[idx].set_ylabel('Frequency', fontsize=10)
        axes[idx].set_title(col.replace('_', ' ').title(), fontsize=11, fontweight='bold')
        axes[idx].set_yscale('log')
        axes[idx].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


In [ ]:
# Text source analysis (which columns contribute most text)
print("=" * 80)
print("TEXT SOURCE CONTRIBUTION ANALYSIS")
print("=" * 80)

source_contribution = {}
for col in text_columns:
    if col in df_text.columns:
        non_empty = df_text[col].dropna()
        non_empty = non_empty[non_empty.astype(str).str.strip() != '']
        
        if len(non_empty) > 0:
            total_chars = non_empty.astype(str).str.len().sum()
            source_contribution[col] = {
                'count': len(non_empty),
                'total_chars': total_chars,
                'avg_chars': total_chars / len(non_empty) if len(non_empty) > 0 else 0
            }

contribution_df = pd.DataFrame(source_contribution).T
contribution_df = contribution_df.sort_values('total_chars', ascending=False)

print("\nText Source Contribution:")
print(contribution_df.to_string())

# Visualize
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

axes[0].barh(contribution_df.index, contribution_df['count'], color='steelblue', alpha=0.7)
axes[0].set_xlabel('Number of Non-Empty Rows', fontsize=11)
axes[0].set_title('Text Source Count', fontsize=12, fontweight='bold')
axes[0].grid(True, alpha=0.3, axis='x')

axes[1].barh(contribution_df.index, contribution_df['total_chars'], color='coral', alpha=0.7)
axes[1].set_xlabel('Total Characters', fontsize=11)
axes[1].set_title('Total Text Contribution', fontsize=12, fontweight='bold')
axes[1].grid(True, alpha=0.3, axis='x')

axes[2].barh(contribution_df.index, contribution_df['avg_chars'], color='green', alpha=0.7)
axes[2].set_xlabel('Average Characters per Row', fontsize=11)
axes[2].set_title('Average Text Length by Source', fontsize=12, fontweight='bold')
axes[2].grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.show()


## Summary and Recommendations for Manipulation Detection

### Key Findings:

1. **TEXT AVAILABILITY:**
   - Combined text available for analysis
   - Multiple text sources (body, title, OCR, ASR)
   - Text length varies significantly

2. **NLP FEATURES:**
   - Word count, sentence structure, punctuation patterns
   - Sentiment analysis (polarity and subjectivity)
   - Emotional indicators (exclamation, questions)

3. **MANIPULATION INDICATORS:**
   - Urgency language
   - Fear appeals
   - Exaggeration patterns
   - Emotional manipulation
   - False authority claims
   - Scarcity tactics
   - Social proof
   - Aggressive calls to action

4. **POLITICAL KEYWORDS:**
   - Election-related terms
   - Party references
   - Policy mentions
   - Issue focus
   - Action words
   - Emotional/polarizing language

### Recommendations for Model Development:

1. **FEATURE ENGINEERING:**
   - Combine text features with metadata (from other dataset)
   - Create composite manipulation scores
   - Use NLP features as numerical inputs
   - Extract n-grams and key phrases

2. **TEXT ANALYSIS:**
   - Use TF-IDF or word embeddings for text representation
   - Consider transformer models (BERT, RoBERTa) for deep text analysis
   - Analyze sentiment and emotional tone
   - Detect manipulation patterns through regex and ML

3. **COMBINED APPROACH:**
   - Merge text dataset with metadata dataset using ad_id
   - Combine text features with spending, targeting, party data
   - Create multi-modal features (text + metadata)

4. **TARGET VARIABLE:**
   - Define manipulation based on:
     * High manipulation pattern scores
     * Extreme sentiment + high spending
     * Urgency + fear appeals + targeting
     * Expert labeling

5. **MODEL SELECTION:**
   - Text classification models (Logistic Regression, SVM, Neural Networks)
   - Ensemble methods combining text and metadata
   - Deep learning for text (LSTM, Transformers)
   - Feature importance analysis to understand manipulation signals

### Next Steps:
1. Merge text dataset with metadata dataset
2. Create comprehensive feature set (text + metadata)
3. Label manipulation examples (manual or rule-based)
4. Train and evaluate models
5. Analyze feature importance
